In [ ]:
# ============DE BASELINE + CLEANING (tek hücre) ============
import re, json, os
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset

# 1. Setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
batch_size = 16
lang = "de"

# 2. Corpus cache
collection_embeddings = torch.load('baseline/cache/collection_embeddings.pt')
collection_keys = np.load('baseline/cache/collection_keys.npy')
print(f"Collection: {collection_embeddings.shape}")

# 3. E5 modeli
e5_name = 'intfloat/multilingual-e5-large'
e5_tokenizer = AutoTokenizer.from_pretrained(e5_name)
e5_model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 loaded.")

def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

# 4. Tweet cleaning (Exp1'deki ile aynı)
def clean_tweet(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\d+/\d+\s*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 5. Embed helper
def embed_texts(texts, max_len=128, desc=""):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    def tok(ex):
        return e5_tokenizer(ex["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt")
    ds = ds.map(tok, batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for batch in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc=f"Embedding {desc}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = e5_model(**batch)
            embs.append(average_pool(outputs.last_hidden_state, batch['attention_mask']).detach().cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

# 6. DE data
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang)
train_raw, dev_raw = data["train"], data["dev"]
train_labels = np.array(train_raw["pubkey"])
dev_labels = np.array(dev_raw["pubkey"])
print(f"DE: train={len(train_raw)}, dev={len(dev_raw)}")

# 7. Iki versiyon: ham (baseline) + cleaned
train_raw_texts = list(train_raw["text"])
dev_raw_texts = list(dev_raw["text"])
train_clean_texts = [clean_tweet(t) for t in train_raw_texts]
dev_clean_texts = [clean_tweet(t) for t in dev_raw_texts]

# 8. Embed (raw)
print("\n--- Embedding RAW (baseline) ---")
train_emb_raw = embed_texts(train_raw_texts, desc="train raw")
dev_emb_raw = embed_texts(dev_raw_texts, desc="dev raw")

# 9. Embed (cleaned)
print("\n--- Embedding CLEANED ---")
train_emb_clean = embed_texts(train_clean_texts, desc="train clean")
dev_emb_clean = embed_texts(dev_clean_texts, desc="dev clean")

# 10. MRR helper
def mrr_at_5(cos_sims, labels):
    top5_idxs = np.argsort(-cos_sims, axis=1)[:, :5]
    top5_keys = collection_keys[top5_idxs]
    labels_2d = labels[:, None]
    matches_mask = (top5_keys == labels_2d)
    positions = np.argmax(matches_mask, axis=1)
    positions[~matches_mask.any(axis=1)] = -1
    rr = np.zeros_like(positions, dtype=float)
    mask = positions >= 0
    rr[mask] = 1.0 / (positions[mask] + 1)
    return float(rr.mean())

# 11. Skorlar
print("\n===DE RESULTS ===")
results = {}
for label, te, de in [
    ("raw_baseline", train_emb_raw, dev_emb_raw),
    ("cleaning",    train_emb_clean, dev_emb_clean),
]:
    train_score = mrr_at_5((te @ collection_embeddings.T).numpy(), train_labels)
    dev_score   = mrr_at_5((de @ collection_embeddings.T).numpy(), dev_labels)
    print(f"{label:20s}  Train: {train_score:.4f}  Dev: {dev_score:.4f}")
    results[label] = {"train": train_score, "dev": dev_score}

# 12. Save embeddings + log
torch.save(train_emb_raw,   'baseline/cache/train_embeddings_de.pt')
torch.save(dev_emb_raw,     'baseline/cache/dev_embeddings_de.pt')
torch.save(train_emb_clean, 'baseline/cache/train_embeddings_de_clean.pt')
torch.save(dev_emb_clean,   'baseline/cache/dev_embeddings_de_clean.pt')

ts = datetime.now().isoformat()
for label, scores in results.items():
    log = {
        "timestamp": ts,
        "experiment": f"exp5_{label}_de",
        "model": "intfloat/multilingual-e5-large",
        "language": "de",
        "transformation": label,
        "mrr5_train": scores["train"],
        "mrr5_dev": scores["dev"],
        "notes": "DE run on M3, batch=16."
    }
    with open('baseline/results/results.jsonl', 'a') as f:
        f.write(json.dumps(log) + '\n')
print("\nLogged + cached.")

/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Collection: torch.Size([10000, 1024])


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7453.88it/s]


E5 loaded.
EN: train=14977, dev=3905

--- Embedding RAW (baseline) ---


Embedding dev raw: 100%|██████████| 245/245 [04:04<00:00,  1.00it/s]



--- Embedding CLEANED ---


Embedding dev clean: 100%|██████████| 245/245 [1:39:43<00:00, 24.42s/it]    



=== EN RESULTS ===
raw_baseline          Train: 0.5028  Dev: 0.4987
cleaning              Train: 0.5019  Dev: 0.5032

Logged + cached.
